# 🎯 Sistema de Analítica de Ocupación con YOLO26

## Proyecto: Análisis de Ocupación en Eventos

**Objetivo:** Construir un sistema completo que analice videos de eventos/plazas/colas para calcular:
- 📊 Aforo instantáneo y picos de ocupación
- 🗺️ Densidad por zonas configurables
- ⏱️ Tiempos de permanencia aproximados
- 📈 Entradas y salidas por periodo
- 📉 Métricas exportables (CSV/JSON)
- 📺 Dashboard visual con gráficas

**Tecnologías:**
- **YOLO26** (última versión Ultralytics 2026) - Detección de personas
- **ByteTrack** - Seguimiento multi-objeto
- **OpenCV** - Procesamiento de video
- **Matplotlib/Seaborn** - Visualizaciones
- **GPU RTX 4060 Ti** - Aceleración por hardware

---

## 📦 Paso 1: Instalación de Dependencias

⚠️ **IMPORTANTE**: Instalar en este orden exacto para evitar conflictos de dependencias entre NumPy, TensorFlow y PyTorch.

In [ ]:
!pip install "ultralytics>=8.3.0"
!pip install "opencv-python>=4.8.0"
!pip install --upgrade yt-dlp
!pip install "lap>=0.5.12"
!pip install "matplotlib>=3.7.0" seaborn
!pip install "pandas>=2.0.0"
!pip install scipy
!pip install tqdm
!pip install "torch>=2.0.0" torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

print("\n✅ Librerías YOLO/PyTorch instaladas")

!pip install tensorflow==2.10.0 tensorflow-gpu==2.10.0
!pip install "numpy<2.0"

print("✅ TensorFlow instalado con configuración compatible")

## 🔍 Paso 2: Verificación de GPU y Entorno

In [ ]:
import torch
import tensorflow as tf

print("=" * 60)
print("🖥️  VERIFICACIÓN DE HARDWARE Y SOFTWARE")
print("=" * 60)

# PyTorch + CUDA
print(f"\n📦 PyTorch versión: {torch.__version__}")
print(f"🔥 CUDA disponible (PyTorch): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU detectada: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    DEVICE = 'cuda'
else:
    print("⚠️  GPU no detectada - usando CPU")
    DEVICE = 'cpu'

# TensorFlow + GPU
print(f"\n📦 TensorFlow versión: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"🔥 GPUs disponibles (TensorFlow): {len(gpus)}")
if gpus:
    for gpu in gpus:
        print(f"   - {gpu}")
else:
    print("⚠️  TensorFlow no detectó GPU")

print("\n" + "=" * 60)
print("✅ Verificación completada")
print("=" * 60)

## 📚 Paso 3: Imports de Librerías

In [ ]:
# Core libraries
import os
import sys
import json
import csv
import time
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import defaultdict, deque

# Computer Vision & Deep Learning
import cv2
import numpy as np
from ultralytics import YOLO
import torch

# Data Processing
import pandas as pd
from scipy import interpolate

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
import seaborn as sns

# Video Download
import yt_dlp

# Progress bars
from tqdm.notebook import tqdm

# Configure matplotlib para notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ Todas las librerías importadas correctamente")

---

## 🎬 Paso 4: VideoManager - Descarga y Gestión de Videos

Clase para descargar videos de YouTube/URL y gestionar archivos de video local.

In [ ]:
class VideoManager:
    """
    Gestiona descarga y carga de videos desde YouTube/URL o archivos locales.
    """
    
    def __init__(self, cache_dir='videos'):
        """
        Args:
            cache_dir: Directorio para almacenar videos descargados
        """
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)
        
    def download_video(self, url: str, output_name: Optional[str] = None) -> str:
        """
        Descarga video de YouTube u otra URL usando yt-dlp.
        
        Args:
            url: URL del video (YouTube, Vimeo, etc.)
            output_name: Nombre personalizado para el archivo (opcional)
            
        Returns:
            Ruta al archivo de video descargado
        """
        print(f"📥 Descargando video desde: {url}")
        
        if output_name is None:
            output_name = f"video_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        output_path = self.cache_dir / f"{output_name}.%(ext)s"
        
        # Configuración de yt-dlp con opciones anti-403 y mejoradas
        ydl_opts = {
            'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
            'outtmpl': str(output_path),
            'quiet': False,
            'no_warnings': False,
            'merge_output_format': 'mp4',
            # Opciones para evitar error 403
            'http_headers': {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
                'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
                'Accept-Language': 'en-us,en;q=0.5',
                'Sec-Fetch-Mode': 'navigate',
            },
            'extractor_args': {
                'youtube': {
                    'player_client': ['android', 'web'],
                    'player_skip': ['webpage', 'configs'],
                }
            },
            'nocheckcertificate': True,
        }
        
        try:
            # Primero extraer info para obtener el nombre final del archivo
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)
                
                # Obtener el nombre del archivo descargado
                if 'requested_downloads' in info and info['requested_downloads']:
                    downloaded_file = info['requested_downloads'][0]['filepath']
                else:
                    # Construir el nombre esperado
                    ext = info.get('ext', 'mp4')
                    downloaded_file = str(self.cache_dir / f"{output_name}.{ext}")
            
            # Verificar que el archivo existe
            if Path(downloaded_file).exists():
                print(f"✅ Video descargado: {downloaded_file}")
                return str(downloaded_file)
            else:
                # Buscar por patrón como fallback
                possible_files = list(self.cache_dir.glob(f"{output_name}.*"))
                if possible_files:
                    downloaded_file = str(possible_files[0])
                    print(f"✅ Video descargado: {downloaded_file}")
                    return downloaded_file
                else:
                    raise FileNotFoundError(f"No se encontró el archivo descargado en {self.cache_dir}")
                
        except Exception as e:
            print(f"❌ Error al descargar video: {e}")
            raise
    
    def load_video(self, source: str) -> Tuple[cv2.VideoCapture, Dict]:
        """
        Carga un video desde archivo o URL.
        
        Args:
            source: Ruta al archivo de video o URL
            
        Returns:
            Tupla (VideoCapture, info_dict)
        """
        # Si es una URL, descargarla primero
        if source.startswith(('http://', 'https://', 'www.')):
            source = self.download_video(source)
        
        # Validar que el archivo existe
        if not Path(source).exists():
            raise FileNotFoundError(f"Video no encontrado: {source}")
        
        # Abrir video con OpenCV
        cap = cv2.VideoCapture(source)
        
        if not cap.isOpened():
            raise ValueError(f"No se pudo abrir el video: {source}")
        
        # Extraer información del video
        info = {
            'path': source,
            'fps': cap.get(cv2.CAP_PROP_FPS),
            'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
            'total_frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
            'duration_sec': int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS))
        }
        
        print(f"\n📹 Video cargado:")
        print(f"   - Resolución: {info['width']}x{info['height']}")
        print(f"   - FPS: {info['fps']:.2f}")
        print(f"   - Frames totales: {info['total_frames']}")
        print(f"   - Duración: {info['duration_sec']} segundos")
        
        return cap, info

print("✅ VideoManager definido")

## 🗺️ Paso 5: OccupancyZone - Gestión de Zonas y Densidad

In [ ]:
class OccupancyZone:
    """
    Define una zona de ocupación mediante un polígono y calcula métricas.
    """
    
    def __init__(self, name: str, polygon_points: List[Tuple[int, int]], 
                 color: Tuple[int, int, int] = None):
        """
        Args:
            name: Nombre de la zona
            polygon_points: Lista de puntos (x, y) que definen el polígono
            color: Color BGR para visualización (aleatorio si no se especifica)
        """
        self.name = name
        self.polygon = np.array(polygon_points, dtype=np.int32)
        self.color = color if color else tuple(np.random.randint(50, 255, 3).tolist())
        
        # Calcular área del polígono (en píxeles)
        self.area_pixels = cv2.contourArea(self.polygon)
        
        # Métricas
        self.current_occupancy = 0
        self.peak_occupancy = 0
        self.total_entries = 0
        self.total_exits = 0
        self.person_ids_inside = set()
        self.person_entry_times = {}  # {person_id: frame_num}
        
    def is_point_inside(self, point: Tuple[float, float]) -> bool:
        """
        Verifica si un punto está dentro del polígono.
        
        Args:
            point: Tupla (x, y)
            
        Returns:
            True si el punto está dentro
        """
        result = cv2.pointPolygonTest(self.polygon, point, False)
        return result >= 0
    
    def update(self, person_positions: Dict[int, Tuple[float, float]], frame_num: int):
        """
        Actualiza las métricas de la zona basándose en posiciones de personas.
        
        Args:
            person_positions: Dict {person_id: (x, y)}
            frame_num: Número de frame actual
        """
        # Calcular quién está dentro ahora
        currently_inside = set()
        for person_id, position in person_positions.items():
            if self.is_point_inside(position):
                currently_inside.add(person_id)
        
        # Detectar entradas (nuevos IDs)
        new_entries = currently_inside - self.person_ids_inside
        self.total_entries += len(new_entries)
        for person_id in new_entries:
            self.person_entry_times[person_id] = frame_num
        
        # Detectar salidas (IDs que ya no están)
        new_exits = self.person_ids_inside - currently_inside
        self.total_exits += len(new_exits)
        for person_id in new_exits:
            if person_id in self.person_entry_times:
                del self.person_entry_times[person_id]
        
        # Actualizar estado
        self.person_ids_inside = currently_inside
        self.current_occupancy = len(currently_inside)
        self.peak_occupancy = max(self.peak_occupancy, self.current_occupancy)
    
    def get_density(self) -> float:
        """
        Calcula densidad (personas por cada 10,000 píxeles).
        
        Returns:
            Densidad actual
        """
        if self.area_pixels == 0:
            return 0.0
        return (self.current_occupancy / self.area_pixels) * 10000
    
    def get_avg_time_inside(self, current_frame: int, fps: float) -> float:
        """
        Calcula tiempo promedio de permanencia en la zona.
        
        Args:
            current_frame: Frame actual
            fps: Frames por segundo del video
            
        Returns:
            Tiempo promedio en segundos
        """
        if not self.person_entry_times:
            return 0.0
        
        total_time = 0
        for entry_frame in self.person_entry_times.values():
            frames_inside = current_frame - entry_frame
            total_time += frames_inside / fps
        
        return total_time / len(self.person_entry_times)
    
    def draw(self, frame: np.ndarray, alpha: float = 0.3) -> np.ndarray:
        """
        Dibuja la zona en el frame con transparencia.
        
        Args:
            frame: Frame de video
            alpha: Transparencia (0-1)
            
        Returns:
            Frame con zona dibujada
        """
        overlay = frame.copy()
        cv2.fillPoly(overlay, [self.polygon], self.color)
        cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
        cv2.polylines(frame, [self.polygon], True, self.color, 2)
        
        # Etiqueta con nombre de zona
        x, y = self.polygon[0]
        cv2.putText(frame, self.name, (x, y - 10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, self.color, 2)
        
        return frame
    
    def to_dict(self) -> Dict:
        """Convierte métricas a diccionario para exportación."""
        return {
            'zone_name': self.name,
            'occupancy': self.current_occupancy,
            'peak': self.peak_occupancy,
            'density': self.get_density(),
            'entries': self.total_entries,
            'exits': self.total_exits,
            'area_pixels': self.area_pixels
        }

print("✅ OccupancyZone definida")

## 📊 Paso 6: MetricsCollector - Recopilación y Exportación de Datos

In [ ]:
class MetricsCollector:
    """
    Recopila, almacena y exporta métricas de ocupación.
    """
    
    def __init__(self, export_dir='exports'):
        """
        Args:
            export_dir: Directorio para exportar datos
        """
        self.export_dir = Path(export_dir)
        self.export_dir.mkdir(exist_ok=True)
        
        self.metrics_history = []
        self.track_history = {}  # {person_id: {'first_seen': frame, 'last_seen': frame}}
        
    def record_frame(self, frame_num: int, timestamp: float, total_persons: int,
                    zones: List[OccupancyZone], fps: float):
        """
        Registra métricas de un frame.
        
        Args:
            frame_num: Número de frame
            timestamp: Timestamp del video
            total_persons: Total de personas detectadas
            zones: Lista de zonas de ocupación
            fps: FPS del video
        """
        metrics = {
            'frame': frame_num,
            'timestamp': timestamp,
            'datetime': datetime.now().isoformat(),
            'total_persons': total_persons,
            'zones': {}
        }
        
        for zone in zones:
            zone_data = zone.to_dict()
            zone_data['avg_time_inside'] = zone.get_avg_time_inside(frame_num, fps)
            metrics['zones'][zone.name] = zone_data
        
        self.metrics_history.append(metrics)
    
    def update_track(self, person_id: int, frame_num: int):
        """
        Actualiza historial de tracking de una persona.
        
        Args:
            person_id: ID de la persona
            frame_num: Frame actual
        """
        if person_id not in self.track_history:
            self.track_history[person_id] = {
                'first_seen': frame_num,
                'last_seen': frame_num
            }
        else:
            self.track_history[person_id]['last_seen'] = frame_num
    
    def get_person_time(self, person_id: int, fps: float) -> float:
        """
        Calcula tiempo visible de una persona.
        
        Args:
            person_id: ID de la persona
            fps: FPS del video
            
        Returns:
            Tiempo en segundos
        """
        if person_id not in self.track_history:
            return 0.0
        
        track = self.track_history[person_id]
        frames = track['last_seen'] - track['first_seen'] + 1
        return frames / fps
    
    def export_csv(self, filename: Optional[str] = None) -> str:
        """
        Exporta métricas a CSV.
        
        Args:
            filename: Nombre del archivo (auto-generado si no se especifica)
            
        Returns:
            Ruta al archivo exportado
        """
        if filename is None:
            filename = f"metrics_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        
        filepath = self.export_dir / filename
        
        # Aplanar datos para CSV
        rows = []
        for metric in self.metrics_history:
            base_row = {
                'frame': metric['frame'],
                'timestamp': metric['timestamp'],
                'datetime': metric['datetime'],
                'total_persons': metric['total_persons']
            }
            
            # Agregar zonas
            for zone_name, zone_data in metric['zones'].items():
                row = base_row.copy()
                row['zone_name'] = zone_name
                row.update(zone_data)
                rows.append(row)
        
        # Crear DataFrame y exportar
        df = pd.DataFrame(rows)
        df.to_csv(filepath, index=False, encoding='utf-8')
        
        print(f"✅ CSV exportado: {filepath}")
        return str(filepath)
    
    def export_json(self, filename: Optional[str] = None) -> str:
        """
        Exporta resumen completo a JSON.
        
        Args:
            filename: Nombre del archivo
            
        Returns:
            Ruta al archivo exportado
        """
        if filename is None:
            filename = f"session_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        
        filepath = self.export_dir / filename
        
        # Calcular estadísticas agregadas
        summary = {
            'session_info': {
                'datetime': datetime.now().isoformat(),
                'total_frames': len(self.metrics_history),
                'total_tracked_persons': len(self.track_history)
            },
            'metrics_history': self.metrics_history,
            'track_history': {str(k): v for k, v in self.track_history.items()}
        }
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        
        print(f"✅ JSON exportado: {filepath}")
        return str(filepath)
    
    def get_summary_stats(self) -> Dict:
        """
        Calcula estadísticas resumidas de toda la sesión.
        
        Returns:
            Diccionario con estadísticas
        """
        if not self.metrics_history:
            return {}
        
        total_persons_per_frame = [m['total_persons'] for m in self.metrics_history]
        
        stats = {
            'total_frames_analyzed': len(self.metrics_history),
            'unique_persons_tracked': len(self.track_history),
            'avg_occupancy': np.mean(total_persons_per_frame),
            'max_occupancy': np.max(total_persons_per_frame),
            'min_occupancy': np.min(total_persons_per_frame),
        }
        
        return stats

print("✅ MetricsCollector definido")

## 🎯 Paso 7: OccupancyAnalyzer - Sistema Principal de Análisis

La clase principal que integra detección YOLO26, tracking, zonas y métricas.

In [ ]:
class OccupancyAnalyzer:
    """
    Sistema principal de análisis de ocupación con YOLO26, tracking y métricas.
    """
    
    def __init__(self, model_name='yolo26s.pt', confidence=0.5, device='cuda'):
        """
        Args:
            model_name: Modelo YOLO a usar (yolo26n, yolo26s, yolo26m)
            confidence: Umbral de confianza para detecciones
            device: 'cuda' o 'cpu'
        """
        print(f"🚀 Inicializando OccupancyAnalyzer con {model_name}")
        
        # Cargar modelo YOLO26
        self.model = YOLO(model_name)
        self.model.to(device)
        self.confidence = confidence
        self.device = device
        
        # Componentes
        self.zones = []
        self.metrics_collector = MetricsCollector()
        
        # Configuración de visualización
        self.show_bbox = True
        self.show_zones = True
        self.show_trails = True
        self.trail_length = 30
        
        # Heatmap acumulativo
        self.heatmap = None
        
        # Historial de posiciones para trails
        self.position_history = defaultdict(lambda: deque(maxlen=self.trail_length))
        
        print(f"✅ Modelo {model_name} cargado en {device}")
    
    def add_zone(self, zone: OccupancyZone):
        """Agrega una zona de ocupación."""
        self.zones.append(zone)
        print(f"✅ Zona '{zone.name}' agregada")
    
    def _get_person_center(self, bbox: List[float]) -> Tuple[float, float]:
        """
        Calcula el centro inferior del bounding box (posición de pies).
        
        Args:
            bbox: [x1, y1, x2, y2]
            
        Returns:
            (center_x, bottom_y)
        """
        x1, y1, x2, y2 = bbox
        center_x = (x1 + x2) / 2
        bottom_y = y2
        return (center_x, bottom_y)
    
    def _annotate_frame(self, frame: np.ndarray, results, frame_num: int, 
                       fps: float) -> np.ndarray:
        """
        Dibuja anotaciones en el frame.
        
        Args:
            frame: Frame de video
            results: Resultados de YOLO
            frame_num: Número de frame
            fps: FPS del video
            
        Returns:
            Frame anotado
        """
        annotated = frame.copy()
        
        # Dibujar zonas
        if self.show_zones:
            for zone in self.zones:
                annotated = zone.draw(annotated, alpha=0.25)
        
        # Extraer detecciones
        if results[0].boxes is not None:
            boxes = results[0].boxes
            
            for i in range(len(boxes)):
                # Obtener datos
                bbox = boxes.xyxy[i].cpu().numpy()
                conf = float(boxes.conf[i])
                cls = int(boxes.cls[i])
                
                # Solo clase persona (0 en COCO)
                if cls != 0 or conf < self.confidence:
                    continue
                
                # Track ID si está disponible
                track_id = int(boxes.id[i]) if boxes.id is not None else None
                
                # Bounding box
                if self.show_bbox:
                    x1, y1, x2, y2 = map(int, bbox)
                    color = self._get_color_for_id(track_id) if track_id else (0, 255, 0)
                    
                    cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
                    
                    # Label
                    label = f"ID:{track_id} {conf:.2f}" if track_id else f"{conf:.2f}"
                    
                    # Tiempo visible si tenemos track ID
                    if track_id and track_id in self.metrics_collector.track_history:
                        time_visible = self.metrics_collector.get_person_time(track_id, fps)
                        label += f" {time_visible:.1f}s"
                    
                    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                    cv2.rectangle(annotated, (x1, y1 - th - 4), (x1 + tw, y1), color, -1)
                    cv2.putText(annotated, label, (x1, y1 - 2), 
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                
                # Trail (rastro de movimiento)
                if self.show_trails and track_id:
                    center = self._get_person_center(bbox)
                    self.position_history[track_id].append(center)
                    
                    # Dibujar trail
                    if len(self.position_history[track_id]) > 1:
                        points = np.array(list(self.position_history[track_id]), dtype=np.int32)
                        color = self._get_color_for_id(track_id)
                        cv2.polylines(annotated, [points], False, color, 2)
        
        # Panel de información
        annotated = self._draw_info_panel(annotated, frame_num, fps)
        
        return annotated
    
    def _draw_info_panel(self, frame: np.ndarray, frame_num: int, fps: float) -> np.ndarray:
        """
        Dibuja panel de información con métricas en tiempo real.
        
        Args:
            frame: Frame de video
            frame_num: Número de frame
            fps: FPS del video
            
        Returns:
            Frame con panel
        """
        # Fondo semi-transparente
        h, w = frame.shape[:2]
        panel_h = 200 + (len(self.zones) * 70)
        overlay = frame.copy()
        cv2.rectangle(overlay, (10, 10), (350, panel_h), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)
        
        y_pos = 35
        
        # Título
        cv2.putText(frame, "METRICAS DE OCUPACION", (20, y_pos), 
                   cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 2)
        y_pos += 30
        
        # Frame y tiempo
        time_sec = frame_num / fps if fps > 0 else 0
        cv2.putText(frame, f"Frame: {frame_num} | Tiempo: {time_sec:.1f}s", (20, y_pos),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        y_pos += 25
        
        # Total de personas
        total_persons = sum(zone.current_occupancy for zone in self.zones)
        cv2.putText(frame, f"Personas totales: {total_persons}", (20, y_pos),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        y_pos += 30
        
        # Métricas por zona
        for zone in self.zones:
            # Nombre de zona con color
            cv2.putText(frame, f"--- {zone.name} ---", (20, y_pos),
                       cv2.FONT_HERSHEY_DUPLEX, 0.5, zone.color, 2)
            y_pos += 20
            
            # Ocupación
            cv2.putText(frame, f"  Ocupacion: {zone.current_occupancy} (Pico: {zone.peak_occupancy})",
                       (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            y_pos += 18
            
            # Densidad
            cv2.putText(frame, f"  Densidad: {zone.get_density():.2f}", (20, y_pos),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            y_pos += 18
            
            # Entradas/Salidas
            cv2.putText(frame, f"  E/S: {zone.total_entries}/{zone.total_exits}", (20, y_pos),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            y_pos += 25
        
        return frame
    
    def _get_color_for_id(self, track_id: int) -> Tuple[int, int, int]:
        """Genera color consistente para un ID."""
        np.random.seed(int(track_id))
        color = tuple(np.random.randint(50, 255, 3).tolist())
        return color
    
    def _update_heatmap(self, frame_shape: Tuple[int, int], positions: List[Tuple[int, int]]):
        """
        Actualiza heatmap de densidad acumulativo.
        
        Args:
            frame_shape: (height, width)
            positions: Lista de posiciones (x, y)
        """
        if self.heatmap is None:
            self.heatmap = np.zeros(frame_shape[:2], dtype=np.float32)
        
        for x, y in positions:
            # Agregar un "blob" gaussiano en la posición
            x, y = int(x), int(y)
            if 0 <= y < self.heatmap.shape[0] and 0 <= x < self.heatmap.shape[1]:
                cv2.circle(self.heatmap, (x, y), 30, 1, -1)
    
    def analyze_video(self, video_source: str, mode='offline', 
                     save_video=True, frame_skip=1) -> Dict:
        """
        Analiza un video completo.
        
        Args:
            video_source: Ruta al video o URL
            mode: 'offline' (procesar todo) o 'realtime' (mostrar mientras procesa)
            save_video: Guardar video anotado
            frame_skip: Procesar cada N frames (para videos largos)
            
        Returns:
            Diccionario con resultados del análisis
        """
        print(f"\n{'='*60}")
        print(f"🎬 INICIANDO ANÁLISIS DE VIDEO")
        print(f"{'='*60}\n")
        
        # Cargar video
        video_manager = VideoManager()
        cap, video_info = video_manager.load_video(video_source)
        
        fps = video_info['fps']
        width = video_info['width']
        height = video_info['height']
        total_frames = video_info['total_frames']
        
        # VideoWriter si guardamos video
        writer = None
        if save_video:
            output_path = self.metrics_collector.export_dir / f"annotated_{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp4"
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))
            print(f"📹 Video anotado se guardará en: {output_path}")
        
        # Progreso
        pbar = tqdm(total=total_frames, desc="Procesando video")
        
        frame_num = 0
        
        try:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                
                # Skip frames si es necesario
                if frame_num % frame_skip != 0:
                    frame_num += 1
                    pbar.update(1)
                    continue
                
                # YOLO tracking (incluye detección + tracking)
                results = self.model.track(
                    frame, 
                    persist=True,
                    conf=self.confidence,
                    classes=[0],  # Solo personas
                    verbose=False,
                    tracker='bytetrack.yaml'
                )
                
                # Extraer posiciones de personas
                person_positions = {}
                if results[0].boxes is not None and results[0].boxes.id is not None:
                    boxes = results[0].boxes
                    for i in range(len(boxes)):
                        track_id = int(boxes.id[i])
                        bbox = boxes.xyxy[i].cpu().numpy()
                        position = self._get_person_center(bbox)
                        person_positions[track_id] = position
                        
                        # Actualizar tracking en metrics
                        self.metrics_collector.update_track(track_id, frame_num)
                        
                        # Actualizar heatmap
                        self._update_heatmap((height, width), [position])
                
                # Actualizar zonas
                for zone in self.zones:
                    zone.update(person_positions, frame_num)
                
                # Registrar métricas
                self.metrics_collector.record_frame(
                    frame_num, 
                    frame_num / fps,
                    len(person_positions),
                    self.zones,
                    fps
                )
                
                # Anotar frame
                annotated_frame = self._annotate_frame(frame, results, frame_num, fps)
                
                # Guardar frame
                if writer:
                    writer.write(annotated_frame)
                
                # Mostrar en tiempo real
                if mode == 'realtime':
                    cv2.imshow('Analisis de Ocupacion', annotated_frame)
                    key = cv2.waitKey(1) & 0xFF
                    if key == ord('q'):
                        print("\n⏹️  Análisis detenido por el usuario")
                        break
                    elif key == ord('p'):
                        cv2.waitKey(-1)  # Pausar
                
                frame_num += 1
                pbar.update(1)
        
        finally:
            pbar.close()
            cap.release()
            if writer:
                writer.release()
            if mode == 'realtime':
                cv2.destroyAllWindows()
        
        print(f"\n✅ Análisis completado: {frame_num} frames procesados")
        
        # Generar resumen
        summary = {
            'video_info': video_info,
            'frames_analyzed': frame_num,
            'stats': self.metrics_collector.get_summary_stats(),
            'zones': [zone.to_dict() for zone in self.zones]
        }
        
        return summary

print("✅ OccupancyAnalyzer definido")

## 📈 Paso 8: Dashboard de Visualización con Matplotlib

Funciones para generar dashboard completo con gráficas de métricas.

In [ ]:
def generate_dashboard(analyzer: OccupancyAnalyzer, video_info: Dict, 
                      save_path: Optional[str] = None):
    """
    Genera dashboard completo con 6 gráficas de análisis.
    
    Args:
        analyzer: Instancia de OccupancyAnalyzer con datos
        video_info: Información del video
        save_path: Ruta para guardar imagen (opcional)
    """
    metrics_history = analyzer.metrics_collector.metrics_history
    
    if not metrics_history:
        print("❌ No hay datos para generar dashboard")
        return
    
    # Configurar estilo
    sns.set_style("whitegrid")
    
    # Crear figura con grid
    fig = plt.figure(figsize=(20, 12))
    gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)
    
    # Extraer datos
    frames = [m['frame'] for m in metrics_history]
    timestamps = [m['timestamp'] for m in metrics_history]
    total_persons = [m['total_persons'] for m in metrics_history]
    
    fps = video_info['fps']
    
    # ========================================
    # GRÁFICA 1: Ocupación Total vs Tiempo
    # ========================================
    ax1 = fig.add_subplot(gs[0, :2])
    ax1.plot(timestamps, total_persons, linewidth=2, color='#2E86AB', label='Ocupación')
    ax1.fill_between(timestamps, total_persons, alpha=0.3, color='#2E86AB')
    ax1.set_xlabel('Tiempo (segundos)', fontsize=12)
    ax1.set_ylabel('Número de Personas', fontsize=12)
    ax1.set_title('📊 Ocupación Total a lo Largo del Tiempo', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Línea de promedio
    avg_occupancy = np.mean(total_persons)
    ax1.axhline(y=avg_occupancy, color='red', linestyle='--', 
               label=f'Promedio: {avg_occupancy:.1f}', linewidth=2)
    ax1.legend()
    
    # ========================================
    # GRÁFICA 2: Ocupación por Zona
    # ========================================
    ax2 = fig.add_subplot(gs[0, 2])
    
    if analyzer.zones:
        zone_names = []
        zone_peaks = []
        zone_colors = []
        
        for zone in analyzer.zones:
            zone_names.append(zone.name)
            zone_peaks.append(zone.peak_occupancy)
            # Convertir BGR a RGB y normalizar
            color_rgb = tuple([c/255 for c in zone.color[::-1]])
            zone_colors.append(color_rgb)
        
        bars = ax2.barh(zone_names, zone_peaks, color=zone_colors, alpha=0.7, edgecolor='black')
        ax2.set_xlabel('Pico de Ocupación', fontsize=12)
        ax2.set_title('🏆 Pico por Zona', fontsize=12, fontweight='bold')
        ax2.grid(True, axis='x', alpha=0.3)
        
        # Valores en barras
        for i, (bar, val) in enumerate(zip(bars, zone_peaks)):
            ax2.text(val + 0.5, bar.get_y() + bar.get_height()/2, 
                    str(val), va='center', fontsize=10, fontweight='bold')
    
    # ========================================
    # GRÁFICA 3: Líneas de Ocupación por Zona
    # ========================================
    ax3 = fig.add_subplot(gs[1, :])
    
    if analyzer.zones:
        for zone in analyzer.zones:
            zone_occupancy = []
            for metric in metrics_history:
                if zone.name in metric['zones']:
                    zone_occupancy.append(metric['zones'][zone.name]['occupancy'])
                else:
                    zone_occupancy.append(0)
            
            color_rgb = tuple([c/255 for c in zone.color[::-1]])
            ax3.plot(timestamps, zone_occupancy, linewidth=2, 
                    label=zone.name, color=color_rgb)
        
        ax3.set_xlabel('Tiempo (segundos)', fontsize=12)
        ax3.set_ylabel('Ocupación', fontsize=12)
        ax3.set_title('📈 Evolución de Ocupación por Zona', fontsize=14, fontweight='bold')
        ax3.legend(loc='upper right')
        ax3.grid(True, alpha=0.3)
    
    # ========================================
    # GRÁFICA 4: Heatmap de Densidad Espacial
    # ========================================
    ax4 = fig.add_subplot(gs[2, 0])
    
    if analyzer.heatmap is not None:
        # Normalizar y aplicar colormap
        heatmap_normalized = cv2.normalize(analyzer.heatmap, None, 0, 255, cv2.NORM_MINMAX)
        heatmap_colored = cv2.applyColorMap(heatmap_normalized.astype(np.uint8), cv2.COLORMAP_JET)
        heatmap_rgb = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
        
        ax4.imshow(heatmap_rgb, aspect='auto')
        ax4.set_title('🔥 Heatmap de Densidad', fontsize=12, fontweight='bold')
        ax4.axis('off')
    
    # ========================================
    # GRÁFICA 5: Histograma de Tiempos
    # ========================================
    ax5 = fig.add_subplot(gs[2, 1])
    
    track_history = analyzer.metrics_collector.track_history
    if track_history:
        times = [analyzer.metrics_collector.get_person_time(pid, fps) 
                for pid in track_history.keys()]
        
        ax5.hist(times, bins=20, color='#A23B72', alpha=0.7, edgecolor='black')
        ax5.set_xlabel('Tiempo Visible (segundos)', fontsize=12)
        ax5.set_ylabel('Frecuencia', fontsize=12)
        ax5.set_title('⏱️ Distribución de Tiempos', fontsize=12, fontweight='bold')
        ax5.grid(True, axis='y', alpha=0.3)
        
        # Línea de promedio
        avg_time = np.mean(times)
        ax5.axvline(x=avg_time, color='red', linestyle='--', 
                   linewidth=2, label=f'Promedio: {avg_time:.1f}s')
        ax5.legend()
    
    # ========================================
    # GRÁFICA 6: Tabla de Estadísticas
    # ========================================
    ax6 = fig.add_subplot(gs[2, 2])
    ax6.axis('off')
    
    # Recopilar estadísticas
    stats_data = []
    stats_data.append(['Métrica', 'Valor'])
    stats_data.append(['─' * 25, '─' * 15])
    
    # Estadísticas generales
    stats = analyzer.metrics_collector.get_summary_stats()
    stats_data.append(['Frames Analizados', f"{stats.get('total_frames_analyzed', 0):,}"])
    stats_data.append(['Personas Únicas', f"{stats.get('unique_persons_tracked', 0)}"])
    stats_data.append(['Ocupación Promedio', f"{stats.get('avg_occupancy', 0):.1f}"])
    stats_data.append(['Pico Global', f"{stats.get('max_occupancy', 0)}"])
    stats_data.append(['Duración Video', f"{video_info.get('duration_sec', 0)}s"])
    
    # Estadísticas por zona
    if analyzer.zones:
        stats_data.append(['', ''])
        stats_data.append(['=== POR ZONA ===', ''])
        for zone in analyzer.zones:
            stats_data.append([f"{zone.name}:", ''])
            stats_data.append([f"  Pico", f"{zone.peak_occupancy}"])
            stats_data.append([f"  Entradas", f"{zone.total_entries}"])
            stats_data.append([f"  Salidas", f"{zone.total_exits}"])
    
    # Crear tabla
    table = ax6.table(cellText=stats_data, cellLoc='left', loc='center',
                     colWidths=[0.6, 0.4])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    
    # Estilos de tabla
    for i in range(len(stats_data)):
        cell = table[(i, 0)]
        if i == 0:
            cell.set_facecolor('#2E86AB')
            cell.set_text_props(weight='bold', color='white')
        elif i % 2 == 0:
            cell.set_facecolor('#E8F4F8')
        
        cell2 = table[(i, 1)]
        if i == 0:
            cell2.set_facecolor('#2E86AB')
            cell2.set_text_props(weight='bold', color='white')
        elif i % 2 == 0:
            cell2.set_facecolor('#E8F4F8')
    
    ax6.set_title('📋 Resumen Estadístico', fontsize=12, fontweight='bold', pad=20)
    
    # Título general
    fig.suptitle('🎯 DASHBOARD DE ANALÍTICA DE OCUPACIÓN - YOLO26', 
                fontsize=18, fontweight='bold', y=0.98)
    
    # Guardar si se especifica
    if save_path is None:
        save_path = analyzer.metrics_collector.export_dir / f"dashboard_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"✅ Dashboard guardado: {save_path}")
    
    plt.show()

print("✅ Función generate_dashboard definida")

## 🖱️ Paso 9: Herramienta Interactiva para Definir Zonas

Utilidad para definir zonas mediante clics en el frame de video.

In [ ]:
class ZoneDefiner:
    """
    Herramienta interactiva para definir zonas en un frame de video.
    """
    
    def __init__(self, frame: np.ndarray):
        """
        Args:
            frame: Frame de video donde definir zonas
        """
        self.frame = frame.copy()
        self.display_frame = frame.copy()
        self.zones_defined = []
        self.current_points = []
        self.current_zone_name = ""
        
    def _mouse_callback(self, event, x, y, flags, param):
        """Callback para eventos del mouse."""
        if event == cv2.EVENT_LBUTTONDOWN:
            # Agregar punto
            self.current_points.append((x, y))
            print(f"  Punto {len(self.current_points)}: ({x}, {y})")
            
            # Dibujar punto
            cv2.circle(self.display_frame, (x, y), 5, (0, 255, 0), -1)
            
            # Dibujar líneas entre puntos
            if len(self.current_points) > 1:
                cv2.line(self.display_frame, 
                        self.current_points[-2], 
                        self.current_points[-1], 
                        (0, 255, 0), 2)
            
            cv2.imshow('Definir Zonas', self.display_frame)
    
    def define_zones_interactive(self, config_save_path: Optional[str] = None) -> List[OccupancyZone]:
        """
        Interfaz interactiva para definir múltiples zonas.
        
        Controles:
            - Click izquierdo: Agregar punto al polígono
            - 'c': Completar zona actual
            - 'n': Nueva zona
            - 'r': Reiniciar zona actual
            - 'q': Terminar y guardar
        
        Args:
            config_save_path: Ruta para guardar configuración JSON
            
        Returns:
            Lista de zonas definidas
        """
        print("\n" + "="*60)
        print("🖱️  DEFINICIÓN INTERACTIVA DE ZONAS")
        print("="*60)
        print("\nControles:")
        print("  Click Izquierdo - Agregar punto al polígono")
        print("  'c' - Completar zona actual")
        print("  'n' - Nueva zona")
        print("  'r' - Reiniciar zona actual")
        print("  'q' - Terminar y guardar")
        print("="*60 + "\n")
        
        cv2.namedWindow('Definir Zonas')
        cv2.setMouseCallback('Definir Zonas', self._mouse_callback)
        
        zone_counter = 1
        
        while True:
            cv2.imshow('Definir Zonas', self.display_frame)
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('c'):  # Completar zona
                if len(self.current_points) >= 3:
                    # Cerrar polígono
                    cv2.line(self.display_frame, 
                            self.current_points[-1], 
                            self.current_points[0], 
                            (0, 255, 0), 2)
                    
                    # Solicitar nombre de zona
                    if not self.current_zone_name:
                        zone_name = f"Zona_{zone_counter}"
                    else:
                        zone_name = self.current_zone_name
                    
                    # Crear zona
                    zone = OccupancyZone(zone_name, self.current_points)
                    self.zones_defined.append(zone)
                    
                    print(f"✅ Zona '{zone_name}' definida con {len(self.current_points)} puntos")
                    
                    # Dibujar zona en el display
                    self.display_frame = zone.draw(self.display_frame, alpha=0.3)
                    
                    # Reset para siguiente zona
                    self.current_points = []
                    self.current_zone_name = ""
                    zone_counter += 1
                else:
                    print("❌ Se necesitan al menos 3 puntos para definir una zona")
            
            elif key == ord('n'):  # Nueva zona
                if self.current_points:
                    print("⚠️  Zona actual no completada. Presiona 'c' para completar o 'r' para reiniciar")
                else:
                    # Solicitar nombre
                    zone_name = input(f"\nNombre para la zona {zone_counter} (Enter para 'Zona_{zone_counter}'): ").strip()
                    self.current_zone_name = zone_name if zone_name else f"Zona_{zone_counter}"
                    print(f"📝 Definiendo zona: {self.current_zone_name}")
                    print("   Click en el frame para agregar puntos...")
            
            elif key == ord('r'):  # Reiniciar zona actual
                if self.current_points:
                    print(f"🔄 Reiniciando zona actual ({len(self.current_points)} puntos descartados)")
                    self.current_points = []
                    self.current_zone_name = ""
                    # Redibujar frame
                    self.display_frame = self.frame.copy()
                    for zone in self.zones_defined:
                        self.display_frame = zone.draw(self.display_frame, alpha=0.3)
            
            elif key == ord('q'):  # Terminar
                if self.current_points:
                    print("\n⚠️  Zona actual no completada. Presiona 'c' primero o 'r' para descartar")
                else:
                    break
        
        cv2.destroyAllWindows()
        
        print(f"\n✅ Total de zonas definidas: {len(self.zones_defined)}")
        
        # Guardar configuración si se especifica
        if config_save_path and self.zones_defined:
            self._save_config(config_save_path)
        
        return self.zones_defined
    
    def _save_config(self, filepath: str):
        """Guarda configuración de zonas en JSON."""
        config = {
            'zones': [
                {
                    'name': zone.name,
                    'polygon': zone.polygon.tolist(),
                    'color': [int(c) for c in zone.color]
                }
                for zone in self.zones_defined
            ],
            'created_at': datetime.now().isoformat()
        }
        
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(config, f, indent=2)
        
        print(f"💾 Configuración de zonas guardada: {filepath}")
    
    @staticmethod
    def load_zones_from_config(filepath: str) -> List[OccupancyZone]:
        """
        Carga zonas desde archivo JSON.
        
        Args:
            filepath: Ruta al archivo de configuración
            
        Returns:
            Lista de zonas
        """
        with open(filepath, 'r', encoding='utf-8') as f:
            config = json.load(f)
        
        zones = []
        for zone_config in config['zones']:
            zone = OccupancyZone(
                name=zone_config['name'],
                polygon_points=zone_config['polygon'],
                color=tuple(zone_config['color'])
            )
            zones.append(zone)
        
        print(f"✅ {len(zones)} zonas cargadas desde {filepath}")
        return zones

print("✅ ZoneDefiner definido")

---

# 🚀 EJEMPLOS DE USO

A continuación se muestran ejemplos completos de cómo usar el sistema de analítica de ocupación.

## 📝 Ejemplo 1: Análisis Básico con Zonas Predefinidas

Análisis de un video con zonas definidas manualmente (sin interacción).

In [ ]:
# ============================================
# CONFIGURACIÓN
# ============================================

# URL del video a analizar (YouTube o archivo local)
VIDEO_URL = "https://www.youtube.com/watch?v=bjafMrpLi8E"
# O usar archivo local:
# VIDEO_PATH = "videos/mi_video.mp4"

# Definir zonas manualmente (coordenadas de polígonos)
# Formato: [(x1, y1), (x2, y2), (x3, y3), ...]
ZONE_ENTRADA = [(100, 300), (300, 300), (300, 500), (100, 500)]
ZONE_CENTRO = [(300, 200), (600, 200), (600, 600), (300, 600)]
ZONE_SALIDA = [(600, 300), (800, 300), (800, 500), (600, 500)]

# ============================================
# PASO 1: Inicializar Analyzer
# ============================================

analyzer = OccupancyAnalyzer(
    model_name='yolo26s.pt',  # Modelo YOLO26 Small (balanceado)
    confidence=0.5,            # Confianza mínima 50%
    device=DEVICE              # Usar GPU si está disponible
)

# ============================================
# PASO 2: Agregar Zonas
# ============================================

analyzer.add_zone(OccupancyZone("Entrada", ZONE_ENTRADA, color=(0, 255, 0)))
analyzer.add_zone(OccupancyZone("Centro", ZONE_CENTRO, color=(255, 0, 0)))
analyzer.add_zone(OccupancyZone("Salida", ZONE_SALIDA, color=(0, 0, 255)))

# ============================================
# PASO 3: Analizar Video
# ============================================

# Modo 'offline': procesa todo el video sin mostrar en tiempo real
# Modo 'realtime': muestra video mientras procesa (presiona 'q' para salir)

results = analyzer.analyze_video(
    video_source=VIDEO_URL,  # O VIDEO_PATH si es local
    mode='offline',          # Cambiar a 'realtime' para ver en vivo
    save_video=True,         # Guardar video anotado
    frame_skip=1             # Procesar todos los frames (mayor = más rápido)
)

print("\n" + "="*60)
print("📊 RESUMEN DE RESULTADOS")
print("="*60)
print(json.dumps(results['stats'], indent=2))

# ============================================
# PASO 4: Exportar Datos
# ============================================

# Exportar CSV con métricas frame a frame
csv_path = analyzer.metrics_collector.export_csv()

# Exportar JSON con resumen completo
json_path = analyzer.metrics_collector.export_json()

# ============================================
# PASO 5: Generar Dashboard
# ============================================

generate_dashboard(analyzer, results['video_info'])

## 📝 Ejemplo 2: Análisis con Definición Interactiva de Zonas

Usa la herramienta interactiva para definir zonas con clics en el video.

In [ ]:
# ============================================
# PASO 1: Descargar/Cargar Video
# ============================================

VIDEO_URL = "https://www.youtube.com/watch?v=bjafMrpLi8E"  # ⚠️ CAMBIAR POR URL REAL

video_manager = VideoManager()
cap, video_info = video_manager.load_video(VIDEO_URL)

# Obtener primer frame para definir zonas
ret, first_frame = cap.read()
cap.release()

if not ret:
    raise ValueError("No se pudo leer el primer frame del video")

print(f"✅ Frame obtenido: {first_frame.shape}")

# ============================================
# PASO 2: Definir Zonas Interactivamente
# ============================================

zone_definer = ZoneDefiner(first_frame)

# Abrir interfaz interactiva
# Instrucciones:
#   - Click para agregar puntos
#   - 'n' para empezar nueva zona
#   - 'c' para completar zona actual
#   - 'r' para reiniciar zona actual
#   - 'q' para terminar

zones = zone_definer.define_zones_interactive(
    config_save_path='zones/mi_configuracion.json'
)

# ============================================
# PASO 3: Analizar con Zonas Definidas
# ============================================

analyzer = OccupancyAnalyzer(
    model_name='yolo26s.pt',
    confidence=0.5,
    device=DEVICE
)

# Agregar zonas definidas
for zone in zones:
    analyzer.add_zone(zone)

# Analizar en tiempo real
results = analyzer.analyze_video(
    video_source=VIDEO_URL,
    mode='realtime',  # Mostrar mientras procesa
    save_video=True,
    frame_skip=1
)

# ============================================
# PASO 4: Exportar y Visualizar
# ============================================

analyzer.metrics_collector.export_csv()
analyzer.metrics_collector.export_json()

generate_dashboard(analyzer, results['video_info'])